<a href="https://colab.research.google.com/github/debo-ogunnowo/Prompt-Inference-System/blob/main/p_reconstructor_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers==4.55.2 peft==0.17.0 accelerate==1.10.0 trl==0.21.0 bitsandbytes==0.47.0 datasets==4.0.0 huggingface-hub==0.34.4 safetensors==0.6.2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/FYP/prompt_repsonse_clean.csv')
df.head()

,message,cleaned_response
0,I think I'm mixing metaphors in this paragraph...,"You're close, but mixing metaphors can be tric..."
1,I keep restating my research question in almos...,You can condense the discussion to reduce redu...
2,Can you weave a reference to Table 2 into this...,"Here's a revised paragraph: ""As observed in th..."
3,"This reminder email buries the actual ask, can...",Here's a rewritten version with a clearer call...
4,These items in the sentence don't have paralle...,You're correct that the sentence has non-paral...


In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
test_df.to_csv('test.csv', index=False)

In [ ]:
# Final model eval
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

repo_id = 'microsoft/Phi-3-mini-4k-instruct'

adapter_path = '/content/drive/MyDrive/FYP/prompt_reconstruction_engine'

tokenizer = AutoTokenizer.from_pretrained(repo_id)
base_model = AutoModelForCausalLM.from_pretrained(
    repo_id,
    torch_dtype=torch.bfloat16,
    device_map='auto'
)

model = PeftModel.from_pretrained(
    base_model,
    adapter_path,
    is_trainable=False
)
model = model.merge_and_unload()
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLU()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=3072, out_features=32064, 

In [ ]:
def reconstruct_prompt(response, max_new_tokens=100):
  messages = [{"role": "user", "content": response}]
  input_text = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True
  )
  inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
  with torch.no_grad():
    outputs = model.generate(
        **inputs,
          max_new_tokens=max_new_tokens,
          do_sample=False,
          temperature=1.0,
          pad_token_id=tokenizer.eos_token_id
    )
  # Decode only newly generated tokens
  generated = outputs[0][inputs['input_ids'].shape[1]:]
  return tokenizer.decode(generated, skip_special_tokens=True).strip()

In [ ]:
test_df = pd.read_csv('test.csv')
test_df.head()

,message,cleaned_response
0,What do you know about Korsakoff's syndrome,Korsakoff's syndrome is a neurological disorde...
1,"search engine indexing, how does that work beh...",Search engine indexing is a complex process th...
2,Compare the strengths and weaknesses of parlia...,Comparative Analysis of Parliamentary and Pres...
3,how do you classify different types of energy ...,Energy sources can be classified into several ...
4,How do mangrove trees survive in saline coasta...,Mangroves are highly adapted trees that thrive...


In [ ]:
reconstructed = []
for i, row in test_df.iterrows():
  inferred = reconstruct_prompt(row['cleaned_response'])
  reconstructed.append(inferred)
  if (i + 1) % 10 == 0:
    print(f"Processed {i + 1}/{len(test_df)}")

test_df['reconstructed_prompt'] = reconstructed

test_df.to_csv('prompt_reconstruction_results.csv', index=False)
print('Prompt Reconstruction Complete. Results Saved')


Processed 10/229
Processed 20/229
Processed 30/229
Processed 40/229
Processed 50/229
Processed 60/229
Processed 70/229
Processed 80/229
Processed 90/229
Processed 100/229
Processed 110/229
Processed 120/229
Processed 130/229
Processed 140/229
Processed 150/229
Processed 160/229
Processed 170/229
Processed 180/229
Processed 190/229
Processed 200/229
Processed 210/229
Processed 220/229
Prompt Reconstruction Complete. Results Saved


In [ ]:
df1 = pd.read_csv('prompt_reconstruction_results.csv')
df1.head()

,message,cleaned_response,reconstructed_prompt
0,What do you know about Korsakoff's syndrome,Korsakoff's syndrome is a neurological disorde...,What is Korsakoff's syndrome and what are its ...
1,"search engine indexing, how does that work beh...",Search engine indexing is a complex process th...,explain how search engine indexing works behin...
2,Compare the strengths and weaknesses of parlia...,Comparative Analysis of Parliamentary and Pres...,Compare and contrast the parliamentary and pre...
3,how do you classify different types of energy ...,Energy sources can be classified into several ...,Classify different types of energy sources
4,How do mangrove trees survive in saline coasta...,Mangroves are highly adapted trees that thrive...,How do mangrove trees survive in saline coasta...


In [ ]:
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
import numpy as np

print('Sample Pairs: ')
for i, row in df1.sample(n=5).iterrows():
    print(f"\nOriginal: {row['message']}")
    print(f"Reconstructed: {row['reconstructed_prompt']}")

Sample Pairs: 

Original: how do solar panels turn sunlight into electricity, just give me a high level step by step
Reconstructed: Explain how solar panels turn sunlight into electricity in a high-level step-by-step explanation

Original: Write an ethics committee report about the academic misconduct of a plagiarist
Reconstructed: Write an ethics committee report on academic misconduct by a faculty member

Original: Can you rewrite this paragraph to make it more suitable for a university level audience: Inflation basically means things get more expensive over time. The government tries to control it by changing interest rates and stuff like that.
Reconstructed: Inflation is a complex macroeconomic phenomenon characterized by a persistent and sustained increase in the general price level of goods and services in an economy over time. This phenomenon is attributed to a variety of factors, including changes in aggregate demand, supply side constraints, and monetary policy interventions. 

In [ ]:
# Eval 1: Cosine similarity

print("\n Evaluating semantic similarity...")
sem_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

original_embeddings = sem_model.encode(
    df1['message'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True
)

reconstructed_embeddings = sem_model.encode(
    df1['reconstructed_prompt'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True
)

similarity_scores = util.cos_sim(
    original_embeddings,
    reconstructed_embeddings
).diagonal().cpu().numpy()

df1['semantic similarity'] = similarity_scores

print(f"Mean semantic similarity:       {similarity_scores.mean():.4f}")
print(f"Median semantic similarity:     {np.median(similarity_scores):.4f}")
print(f"Proportion above 0.7:           {(similarity_scores > 0.7).mean():.4f}")
print(f"Proportion above 0.5:           {(similarity_scores > 0.5).mean():.4f}")


 Evaluating semantic similarity...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Mean semantic similarity:       0.8255
Median semantic similarity:     0.8593
Proportion above 0.7:           0.8384
Proportion above 0.5:           0.9563


In [ ]:
df1.head()

,message,cleaned_response,reconstructed_prompt,semantic similarity
0,What do you know about Korsakoff's syndrome,Korsakoff's syndrome is a neurological disorde...,What is Korsakoff's syndrome and what are its ...,0.921744
1,"search engine indexing, how does that work beh...",Search engine indexing is a complex process th...,explain how search engine indexing works behin...,0.977461
2,Compare the strengths and weaknesses of parlia...,Comparative Analysis of Parliamentary and Pres...,Compare and contrast the parliamentary and pre...,0.903303
3,how do you classify different types of energy ...,Energy sources can be classified into several ...,Classify different types of energy sources,0.974546
4,How do mangrove trees survive in saline coasta...,Mangroves are highly adapted trees that thrive...,How do mangrove trees survive in saline coasta...,0.930455


In [ ]:
# Eval 2: Category Agreement
import torch

print("\n Evaluating category agreement...")
label_map = {
    'LABEL_0': 'Research&Inquiry',
    'LABEL_1': 'Content Generation',
    'LABEL_2': 'Text Refinement'
}

classifier = pipeline(
    'text-classification',
   # model='/content/drive/MyDrive/FYP/prompt_classifier',
   # tokenizer='/content/drive/MyDrive/FYP/prompt_classifier',
    model='/content/drive/MyDrive/prompt_classifier_v2/final_model',
    tokenizer='/content/drive/MyDrive/prompt_classifier_v2/final_model',
    device=0 if torch.cuda.is_available() else -1
)

def get_category(text):
  try:
    result = classifier(
      str(text),
      truncation=True,
      max_length=128
    )[0]
    return label_map[result['label']]
  except Exception as e:
    return 'Unknown'

df1['original_category'] = df1['message'].apply(get_category)
df1['reconstructed_category'] = df1['reconstructed_prompt'].apply(get_category)
df1['category_match'] = (
    df1['original_category'] == df1['reconstructed_category']
)

agreement_rate = df1['category_match'].mean()
print(f"Overall category agreement rate: {agreement_rate:.4f}, {agreement_rate:.2%}")
print("\nAgreement rate by original category:")
print(df1.groupby('original_category')['category_match'].mean())

#df1.to_csv('/content/drive/MyDrive/FYP/final_reconstruction_evaluation_results.csv', index=False)
#print("\nFull evaluation results saved.")




 Evaluating category agreement...


Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Overall category agreement rate: 0.7860, 78.60%

Agreement rate by original category:
original_category
Content Generation    0.700000
Research&Inquiry      0.909091
Text Refinement       0.683333
Name: category_match, dtype: float64
